# Analisis dan Ekstraksi Fitur Time Series Kualitas Udara (NO2, CO, SO2)
**Lokasi:** Kwanyar & Sekitarnya (Kabupaten Bangkalan)
**Rentang Waktu:** 31 Agustus 2025 - 31 Agustus 2026

Tahap pertama dalam *pipeline* ini adalah mempersiapkan lingkungan kerja dengan menginstal library **TSFEL (Time Series Feature Extraction Library)**. Library ini akan digunakan untuk mengekstrak puluhan fitur statistik, spektral, dan temporal dari data ketiga gas polusi (NO2, CO, dan SO2) di wilayah Kwanyar dan sekitarnya.

In [ ]:
!pip install tsfel

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.4/63.4 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 66.5 MB/s eta 0:00:00


### 1. Muat Data dan Pembersihan (Outlier & Missing Values)
Tahap ini bertujuan untuk membaca data historis `KualitasUdara_Kwanyar_Sekitarnya.csv`. Nilai ekstrem (*outliers*) untuk masing-masing polutan (**NO2, CO, dan SO2**) akan dideteksi menggunakan metode IQR (Interquartile Range) lalu diubah menjadi NaN. Setelah itu, seluruh *missing values* akan diisi kembali menggunakan metode interpolasi waktu.

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv('KualitasUdara_Kwanyar_Sekitarnya.csv')
df['date'] = pd.to_datetime(df['date'])
df = df.sort_values('date').reset_index(drop=True)

# Daftar ketiga target polutan yang akan diproses
targets = ['NO2', 'CO', 'SO2']

for target in targets:
    # Paksa kolom target jadi numerik
    df[target] = pd.to_numeric(df[target], errors='coerce')

    n_missing_before = df[target].isna().sum()
    print(f"[{target}] Jumlah missing value awal: {n_missing_before}")

    # Deteksi Outlier dengan IQR
    Q1 = df[target].quantile(0.25)
    Q3 = df[target].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    # Ubah nilai outlier menjadi NaN
    outlier_condition = (df[target] < lower_bound) | (df[target] > upper_bound)
    df.loc[outlier_condition, target] = np.nan

# Imputasi Missing Value & Outlier untuk KETIGA kolom sekaligus menggunakan interpolasi waktu
df_clean = df.set_index('date').interpolate(method='time').ffill().bfill()

print("\nPembersihan selesai! Outlier sudah dihilangkan dan Missing Value sudah diimputasi untuk NO2, CO, dan SO2.")

# --- MENYIMPAN HASIL INTERPOLASI ---
df_clean_csv = df_clean.reset_index()
nama_file_bersih = 'KualitasUdara_Kwanyar_Sekitarnya_Cleaned.csv'
df_clean_csv.to_csv(nama_file_bersih, index=False)

print(f"File hasil interpolasi '{nama_file_bersih}' .")

[NO2] Jumlah missing value awal: 121
[CO] Jumlah missing value awal: 126
[SO2] Jumlah missing value awal: 88

Pembersihan selesai! Outlier sudah dihilangkan dan Missing Value sudah diimputasi untuk NO2, CO, dan SO2.
File hasil interpolasi 'KualitasUdara_Kwanyar_Sekitarnya_Cleaned.csv' .


### 2. Imputasi Missing Value dengan Interpolasi Waktu
Data yang hilang atau *outlier* yang telah dihapus di kolom NO2, CO, maupun SO2 tidak boleh dibiarkan kosong karena akan menggagalkan ekstraksi fitur TSFEL. Untuk mengatasinya, kita menggunakan metode **Interpolasi Waktu (Time Interpolation)**.

Metode ini memperkirakan nilai yang hilang dengan menarik garis lurus secara proporsional antara titik data hari sebelum dan sesudah nilai yang kosong.

**Rumus Linear Interpolation:**
$$y = y_0 + (x - x_0) \frac{y_1 - y_0}{x_1 - x_0}$$

**Keterangan Rumus:**
*   $y$ = Nilai polusi estimasi pada hari yang kosong (target).
*   $x$ = Urutan waktu/indeks hari data yang kosong.
*   $x_0, y_0$ = Waktu dan nilai polusi pada hari **tepat sebelum** data kosong.
*   $x_1, y_1$ = Waktu dan nilai polusi pada hari **tepat sesudah** data kosong.

**Contoh Perhitungan Manual (Kasus Data NO2 Kwanyar & Sekitarnya):**
Berdasarkan data mentah satelit, perekaman sensor gas $NO_2$ terputus (tercatat NaN) pada tanggal **10 September 2025**. Kita akan menghitung estimasi nilainya secara matematis menggunakan data tanggal 9 Sept dan 11 Sept.

*   09 Sept 2025 ($x_0$) = 0.000025465 ($y_0$)
*   10 Sept 2025 ($x$) = NaN ($y$ yang akan dicari)
*   11 Sept 2025 ($x_1$) = 0.000015728 ($y_1$)

Mengingat jarak waktu dari 9 Sept ke 10 Sept adalah 1 hari ($x - x_0 = 1$), dan jarak dari 9 Sept ke 11 Sept adalah 2 hari ($x_1 - x_0 = 2$), maka perhitungannya:

$$y = 0.000025465 + (1) \frac{0.000015728 - 0.000025465}{2}$$
$$y = 0.000025465 + \frac{-0.000009737}{2}$$
$$y = 0.000025465 - 0.0000048685$$

**$$y = 0.000020596$$**

Hasil perhitungan manual ini bernilai **0.000020596**, yang mana terbukti persis dan akurat dengan nilai imputasi otomatis yang ada di file `KualitasUdara_Kawnyar_Sekitarnya_Cleaned.csv` pada baris tanggal 10 September 2025. Proses inilah yang dilakukan library pandas ke seluruh data yang kosong secara massal.

### 3. Ekstraksi Fitur Polutan (TSFEL)
Tahap ini mengekstraksi data deret waktu yang sudah bersih menjadi 68 fitur (statistik, spektral, dan temporal) menggunakan library TSFEL. Ekstraksi dilakukan secara terpisah untuk masing-masing polutan (NO2, CO, dan SO2), sehingga akan menghasilkan tiga file CSV yang berbeda agar analisis per jenis gas lebih terfokus dan akurat.

In [ ]:
import inspect
import pandas as pd
import numpy as np
import tsfel.feature_extraction.features as tsfel_features

# Frekuensi sampling (1 per hari)
fs = 1
targets = ['NO2', 'CO', 'SO2']

# ---------- Daftar 68 fitur dasar ----------
FEATURE_LIST = """abs_energy auc autocorr average_power calc_centroid calc_max calc_mean
calc_median calc_min calc_std calc_var dfa distance ecdf ecdf_percentile ecdf_percentile_count
ecdf_slope entropy fundamental_frequency higuchi_fractal_dimension hist_mode human_range_energy
hurst_exponent interq_range kurtosis lempel_ziv lpcc max_frequency max_power_spectrum
maximum_fractal_length mean_abs_deviation mean_abs_diff mean_diff median_abs_deviation
median_abs_diff median_diff median_frequency mfcc mse negative_turning neighbourhood_peaks
petrosian_fractal_dimension pk_pk_distance positive_turning power_bandwidth rms skewness slope
spectral_centroid spectral_decrease spectral_distance spectral_entropy spectral_kurtosis
spectral_positive_turning spectral_roll_off spectral_roll_on spectral_skewness spectral_slope
spectral_spread spectral_variation spectrogram_mean_coeff sum_abs_diff wavelet_abs_mean
wavelet_energy wavelet_entropy wavelet_std wavelet_var zero_cross""".split()

def to_scalar(result):
    if isinstance(result, dict) and "values" in result:
        result = result["values"]
    if isinstance(result, (list, tuple, np.ndarray)):
        arr = np.asarray(result, dtype=float)
        return float(np.nanmean(arr))
    return float(result)

def extract_one(fn_name, signal, fs):
    fn = getattr(tsfel_features, fn_name)
    params = inspect.signature(fn).parameters
    if "fs" in params:
        result = fn(signal, fs)
    else:
        result = fn(signal)
    return to_scalar(result)

# Lakukan ekstraksi untuk tiap-tiap target polutan secara terpisah
for target in targets:
    signal_1d = df_clean[target].astype(float).values
    row = {}

    # Ekstraksi 68 fitur untuk gas saat ini
    for fn_name in FEATURE_LIST:
        row[fn_name] = extract_one(fn_name, signal_1d, fs)

    # Ubah hasil ekstraksi menjadi DataFrame
    extracted_features = pd.DataFrame([row])

    # Simpan ke CSV dengan nama spesifik per gas
    nama_file_csv = f'{target}_Kwanyar_Sekitarnya_TSFEL.csv'
    extracted_features.to_csv(nama_file_csv, index=False)

    print(f"Berhasil! Ekstraksi {extracted_features.shape[1]} fitur untuk {target} telah disimpan di '{nama_file_csv}'.")

Berhasil! Ekstraksi 68 fitur untuk NO2 telah disimpan di 'NO2_Kwanyar_Sekitarnya_TSFEL.csv'.
Berhasil! Ekstraksi 68 fitur untuk CO telah disimpan di 'CO_Kwanyar_Sekitarnya_TSFEL.csv'.
Berhasil! Ekstraksi 68 fitur untuk SO2 telah disimpan di 'SO2_Kwanyar_Sekitarnya_TSFEL.csv'.




Dalam ekstraksi fitur menggunakan pustaka **TSFEL**, terdapat puluhan fitur yang dihasilkan untuk mendeskripsikan karakteristik polutan. Berikut adalah penjelasan, rumus matematis, dan pembuktian perhitungan manual untuk 2 fitur temporal, yaitu **`neighbourhood_peaks`** dan **`pk_pk_distance`**.

#### Data Sampel (20 Observasi Awal $NO_2$)
Untuk membuktikan komputasi TSFEL, kita menggunakan 20 hari pertama dari data $NO_2$ Samarinda yang telah diinterpolasi:
* $x_{1} = 0.00001846$ | $x_{6} = 0.00000983$ | $x_{11} = 0.00001206$ | $x_{16} = 0.00001079$
* $x_{2} = 0.00001846$ | $x_{7} = 0.00000986$ | $x_{12} = 0.00001123$ | $x_{17} = 0.00001034$
* $x_{3} = 0.00000733$ | $x_{8} = 0.00000526$ | $x_{13} = 0.00000850$ | $x_{18} = 0.00000695$
* $x_{4} = 0.00000894$ | $x_{9} = 0.00001010$ | $x_{14} = 0.00001098$ | $x_{19} = 0.00000311$
* $x_{5} = 0.00000735$ | $x_{10}= 0.00000839$ | $x_{15} = 0.00000939$ | $x_{20} = 0.00000750$

---

#### Fitur 1: neighbourhood_peaks(signal, n)

*   **Deskripsi Fitur:** Fitur temporal ini menghitung **jumlah puncak lokal (*local peaks*)** yang menonjol di dalam rentang data deret waktu. Sebuah titik observasi ($x_i$) diklasifikasikan sebagai "puncak" apabila nilainya **lebih besar** dari $n$ data tetangga di sebelah kirinya dan $n$ data tetangga di sebelah kanannya.
*   **Rumus Matematika:**
$$Peaks = \sum_{i=n}^{N-n} I(x_i > x_{i \pm k}, \forall k \in [1, n])$$
*   **Keterangan Rumus:**
    *   $I$: Fungsi indikator (Menghasilkan angka 1 jika titik tersebut memenuhi syarat puncak, dan 0 jika tidak).
    *   $n$: Parameter lebar jendela tetangga (*neighborhood window*). TSFEL akan mengecek jarak sejauh $n$ titik ke belakang dan ke depan.
    *   $N$: Total panjang data pengamatan.
*   **Contoh Perhitungan Manual (Menggunakan $n = 2$):**
    Kita mencari titik mana di antara 20 sampel di atas yang lebih besar dari 2 hari sebelum **dan** 2 hari sesudahnya. Mari kita evaluasi beberapa kandidat titik lonjakan:
    *   Kandidat 1 ($x_4 = 0.00000894$): Dicek dengan tetangganya ($x_2, x_3$ dan $x_5, x_6$). Ternyata $x_2$ dan $x_6$ memiliki nilai yang lebih besar dari $x_4$. **(Bukan Puncak)**
    *   Kandidat 2 ($x_7 = 0.00000986$): Dicek dengan ($x_5, x_6$ dan $x_8, x_9$). Ternyata data $x_9$ ($0.00001010$) lebih besar dari $x_7$. **(Bukan Puncak)**
    *   Kandidat 3 ($x_{11} = \mathbf{0.00001206}$): Dicek dengan data tetangga ($x_9, x_{10}$ dan $x_{12}, x_{13}$).
        * $x_{11} > x_9$ ($0.00001010$) & $x_{10}$ ($0.00000839$)
        * $x_{11} > x_{12}$ ($0.00001123$) & $x_{13}$ ($0.00000850$)
        * **(Memenuhi Syarat Puncak Valid!)**
    
    *Hasil dari pembuktian manual pada 20 baris pertama untuk $n=2$ adalah:* **`neighbourhood_peaks` = 1**

---

#### Fitur 2: pk_pk_distance(signal)

*   **Deskripsi Fitur:** Fitur statistik ini menghitung **Jarak Peak-to-Peak** (Puncak-ke-Puncak). Fitur ini mengukur selisih absolut antara nilai observasi tertinggi (puncak maksimum) dengan nilai observasi terendah (lembah minimum) dari keseluruhan rangkaian deret waktu. Semakin besar nilainya, semakin drastis fluktuasi gas pencemar di lokasi observasi tersebut.
*   **Rumus Matematika:**
$$Distance_{pk} = \max(x) - \min(x)$$
*   **Keterangan Rumus:**
    *   $\max(x)$: Nilai polusi tertinggi dalam sinyal *time series*.
    *   $\min(x)$: Nilai polusi terendah dalam sinyal *time series*.
*   **Contoh Perhitungan Manual (Berdasarkan 20 sampel):**
    Pertama, kita memindai 20 data sampel $NO_2$ di atas untuk mencari rekor nilai tertinggi dan terendahnya:
    *   Nilai Maksimum ($\max(x)$) jatuh pada indeks ke-1 yaitu $x_1 = 0.00001846$
    *   Nilai Minimum ($\min(x)$) jatuh pada indeks ke-19 yaitu $x_{19} = 0.00000311$
    
    Selanjutnya, operasikan rumusnya:
    $$Distance_{pk} = 0.00001846 - 0.00000311$$
    $$Distance_{pk} = \mathbf{0.00001535}$$
    
    *Hasil akhir pembuktian `pk_pk_distance` untuk 20 sampel tersebut adalah:* **$0.00001535$**